## DeepLabCut Pipeline
 ---
 
Kernel에
deeplabcut
napari-deeplabcut
torch (CUDA 버전)
 다운로드

**Body parts:** nose, R_ear, L_ear, neck, body_center, tail_base, tail_end  

---

## 0. 환경 확인

In [ ]:
import deeplabcut
import torch

print(f"DLC version : {deeplabcut.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"GPU 사용 가능 : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")

## 1. 경로 설정
비디오 파일 및 출력 파일 선택

In [ ]:
import os
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

# ── 비디오 파일 선택 ────────────────────────────────────────
video_file = filedialog.askopenfilename(
    title="비디오 파일을 선택하세요",
    initialdir=os.path.expanduser("~/Desktop"),
    filetypes=[("AVI files", "*.avi"), ("모든 파일", "*.*")]
)

if not video_file:
    root.destroy()
    raise ValueError("비디오 파일을 선택하지 않았습니다.")

# ── OUTPUT_DIR 선택 ──────────────────────────────────────
output_dir = filedialog.askdirectory(
    title="출력 폴더를 선택하세요",
    initialdir=r""
)

root.destroy()

if not output_dir:
    raise ValueError("출력 폴더를 선택하지 않았습니다.")

VIDEO_PATH = video_file
VIDEO_FILENAME = os.path.splitext(os.path.basename(video_file))[0]

# ── BASE_DIR 설정 (비디오 파일의 폴더) ──────────────────────
BASE_DIR = os.path.dirname(VIDEO_PATH)

# ── OUTPUT_DIR 설정 ──────────────────────────────────────
OUTPUT_DIR = output_dir

print(f"BASE_DIR    : {BASE_DIR}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"비디오 파일 : {VIDEO_FILENAME}")
print(f"비디오 경로 : {VIDEO_PATH}")

## 2. 프로젝트 생성

Project 및 실험자 이름 수정 -> 이후에도 통일할 것

In [ ]:
config_path = deeplabcut.create_new_project(
    project="FP_cannula_10112",
    experimenter="SH",
    videos=[VIDEO_PATH],
    working_directory=OUTPUT_DIR,
    copy_videos=False,
    multianimal=False
)

if config_path is None:
    raise ValueError("프로젝트 생성 실패 — 이미 존재합니다. config_path를 직접 입력하세요.")

print(f"\nconfig.yaml 경로:\n{config_path}")

## 3. config.yaml 자동 설정
body parts, skeleton, 추출 프레임 수를 자동으로 업데이트합니다.

In [ ]:
import ruamel.yaml
yaml = ruamel.yaml.YAML()

with open(config_path, 'r') as f:
    cfg = yaml.load(f)

# Body parts 설정
cfg['bodyparts'] = ['nose', 'R_ear', 'L_ear', 'neck', 'body_center', 'tail_base', 'tail_end']

# Skeleton 설정
cfg['skeleton'] = [
    ['nose', 'R_ear'],
    ['nose', 'L_ear'],
    ['nose', 'neck'],
    ['neck', 'body_center'],
    ['body_center', 'tail_base'],
    ['tail_base', 'tail_end']
]

# 추출 프레임 수 (영상 길이에 따라 조정)
cfg['numframes2pick'] = 50

# 시각화 설정
cfg['dotsize'] = 6
cfg['colormap'] = 'rainbow'
cfg['skeleton_color'] = 'white'

with open(config_path, 'w') as f:
    yaml.dump(cfg, f)

print("config.yaml 업데이트 완료!")
print(f"Body parts: {cfg['bodyparts']}")

## 4. 프레임 추출

In [ ]:
deeplabcut.extract_frames(
    config_path,
    mode='automatic',
    algo='uniforms',      # kmeans보다 빠름 (50프레임이면 다양성 충분)
    userfeedback=False
)

print("\n프레임 추출 완료!")
print("labeled-data 폴더에서 추출된 프레임 확인 가능")

## 5. 라벨링
napari 기반 라벨링 툴이 실행됩니다.  
각 프레임에서 7개 body part를 순서대로 클릭하세요.

**순서:** nose → R_ear → L_ear → neck → body_center → tail_base → tail_end

In [ ]:
deeplabcut.label_frames(config_path)

## 6. 라벨 확인
라벨링 결과를 이미지로 저장해서 확인합니다.

In [ ]:
deeplabcut.check_labels(
    config_path,
    visualizeindividuals=True
)

print("\nlabeled-data 폴더 내 _labeled 폴더에서 확인 이미지를 볼 수 있습니다.")